In [123]:
# Newton's method with mixed precisions: ur (residual) <= u (update) <= uℓ (linear solve)
# Example: ur = Float16, u = Float32, ul = Float64

function newton_mixed_precision(x0, ul, u, ur; imax=10, atol=1e-6)

    x = x0

    for i in 1:imax
        # Step 1: compute residual in ur precision
        x = ur(x0)
        f = ur(sin(x) + cos(10*x) + x^5 - exp(-x)) - 2   
        abs_f = abs(Float64(f))
        if abs_f < atol
            println("Converged at iter $(i), x=$(x)")
            break
        end

        # Step 2: solve J(x)d = -f in ul precision
        x = ul(x0)
        J = ul(cos(x) - 10*sin(10*x) + 5x^4 + exp(-x))             # Jacobian at high precision
        rhs = -ul(f)   # convert f to ul
        d = rhs / J             # solve scalar system

        # Step 3: update in u precision
        x = u(u(x) + u(d))
    end

    return x
end

newton_mixed_precision (generic function with 1 method)

In [127]:
start_time = time()
y = newton_mixed_precision(1.0, Float16, Float32, Float64)
elapsed = time() - start_time
println("final = ", y)
println("Run time: ", elapsed)

final = 1.1203613
Run time: 0.0011200904846191406
